<a href="https://colab.research.google.com/github/MalakSalehh/Thesis-datasets/blob/main/FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q openai pandas numpy scipy scikit-learn

In [2]:
import re
import json
import time
import numpy as np
import pandas as pd
from openai import OpenAI

In [3]:
GROQ_API_KEY = "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"

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

In [4]:
url = "https://raw.githubusercontent.com/MalakSalehh/Thesis-datasets/main/training_set_rel3.tsv"
df = pd.read_csv(url, sep="\t", encoding="latin1")
print(df.shape)
df.head()

(12976, 28)


,essay_id,essay_set,essay,rater1_domain1,rater2_domain1,rater3_domain1,domain1_score,rater1_domain2,rater2_domain2,domain2_score,...,rater2_trait3,rater2_trait4,rater2_trait5,rater2_trait6,rater3_trait1,rater3_trait2,rater3_trait3,rater3_trait4,rater3_trait5,rater3_trait6
0,1,1,"Dear local newspaper, I think effects computer...",4,4,NaN,8,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1,"Dear @CAPS1 @CAPS2, I believe that using compu...",5,4,NaN,9,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,1,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",4,3,NaN,7,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,1,"Dear Local Newspaper, @CAPS1 I have found that...",5,5,NaN,10,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,1,"Dear @LOCATION1, I know having computers has a...",4,4,NaN,8,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
set1 = df[df["essay_set"] == 1].copy()
set1 = set1[["essay_id", "essay", "domain1_score"]]

set1["essay"] = set1["essay"].str.replace("\n", " ", regex=False)
set1["essay"] = set1["essay"].str.strip()

print(set1.shape)
set1.head()

(1783, 3)


,essay_id,essay,domain1_score
0,1,"Dear local newspaper, I think effects computer...",8
1,2,"Dear @CAPS1 @CAPS2, I believe that using compu...",9
2,3,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",7
3,4,"Dear Local Newspaper, @CAPS1 I have found that...",10
4,5,"Dear @LOCATION1, I know having computers has a...",8


In [7]:
def categorize(score):
    if score <= 5:
        return "Low"
    elif score <= 8:
        return "Medium"
    else:
        return "High"

set1["score_category"] = set1["domain1_score"].apply(categorize)
set1["domain1_score"].value_counts().sort_index()

,count
domain1_score,
2,10
3,1
4,17
5,17
6,110
7,135
8,687
9,334
10,316


In [8]:
cal_low = set1[set1["score_category"] == "Low"].sample(2, random_state=42)
cal_med = set1[set1["score_category"] == "Medium"].sample(2, random_state=42)
cal_high = set1[set1["score_category"] == "High"].sample(2, random_state=42)

calibration = pd.concat([cal_low, cal_med, cal_high]).sample(frac=1, random_state=42).reset_index(drop=True)
calibration

,essay_id,essay,domain1_score,score_category
0,1592,The computers are cool. Do you now I werpsite ...,2,Low
1,995,"I think computers are good, because some peopl...",4,Low
2,686,"Dear local newspaper, I would like to speak up...",9,High
3,1572,"Dear to @CAPS1 this @MONTH1 concern, Computers...",8,Medium
4,1567,"Dear @ORGANIZATION1, @DATE1, everywhere you lo...",11,High
5,146,Dear local newspaper I think that usieng compu...,6,Medium


In [9]:
calibration_ids = calibration["essay_id"].tolist()
demo = set1[~set1["essay_id"].isin(calibration_ids)].sample(5, random_state=42).reset_index(drop=True)
demo

,essay_id,essay,domain1_score,score_category
0,66,Wouldnt it be great to use computers in everyd...,9,High
1,946,According to the @ORGANIZATION1 govermant @PER...,8,Medium
2,837,"Dear, @ORGANIZATION1 the big rectangular shape...",8,Medium
3,1710,Wouldn't you agree that computers have a great...,9,High
4,804,"Dear Local Newspaper, The effects computers ha...",8,Medium
